## 9. CNN 网络中的正则化 - Dropout&Batch Normalization

#### 1. 为什么 CNN 中也需要正则化

##### 1.1 核心原因
虽然 CNN 通过：
* 局部连接
* 参数共享
* 池化 / 下采样

已经比普通 MLP 更适合图像任务，

但它仍然可能出现这些问题：
* 训练集效果很好，测试集效果差
* 模型记住训练数据细节，泛化能力不够
* 深层网络训练不够稳定

所以在 CNN 中，我们同样会使用一些正则化或稳定训练的方法，

其中最常见的就是：
* Dropout
* Batch Normalization（BatchNorm）

##### 1.2 和 MLP 中的关系
我们之前在 MLP 部分已经详细学过：
* Dropout 的基本思想
* BatchNorm 的作用和实现

所以这一节不再重复展开底层原理，

而是重点说明：

它们在 CNN 中怎么放、怎么理解、和 MLP 有什么使用差别。

#### 2. CNN 中的 Dropout

##### 2.1 基本作用
Dropout 在 CNN 中的核心作用和在 MLP 中一样：

训练时随机让一部分神经元失活，从而减少模型对局部特征的过度依赖。

这样可以帮助模型：
* 减少过拟合
* 提高泛化能力
* 避免某些特征通道被过度依赖

##### 2.2 在 CNN 中的直观理解
在 CNN 里，前面卷积层提取出了很多特征图或高层特征。

如果模型太依赖某些固定特征响应，

那么它在新数据上可能就不够稳。

这时 Dropout 可以理解为：

训练时故意“遮掉”一部分特征响应，让网络学会不要只依赖某几个局部模式。

#### 3. CNN 中 Dropout 一般放在哪里

##### 3.1 更常见于全连接层前后
在比较经典的 CNN 中，Dropout 常见位置是：
* Flatten 之后
* 全连接层之前或之间

例如：

`Conv → Pool → Flatten → FC → Dropout → FC`

这是因为全连接层参数量通常更大，

更容易过拟合，所以更常加 Dropout。

##### 3.2 卷积层后也可以加，但通常更谨慎
在卷积层后面也可以使用 Dropout，

但相比 MLP 的全连接部分，通常会更谨慎一些。

因为卷积层前部负责提取基础局部特征，

如果失活太强，可能会影响特征学习。

所以在基础 CNN 中，最常见的理解是：
* FC 部分更常加 Dropout
* 卷积部分可以加，但通常不是第一优先位置

#### 4. PyTorch 中 CNN 里的 Dropout 写法

##### 4.1 用在全连接层附近
这里最值得先记住的是：
Dropout 常常放在全连接层之间。

In [2]:
import torch.nn as nn

class CNNWithDropout(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

##### 4.2 如果是卷积特征图上使用
在 CNN 中，如果想直接对卷积输出做 dropout，

PyTorch 里还常见：

`nn.Dropout2d()`

它更适合用于特征图场景。

#### 5. CNN 中如何理解 Dropout 的使用

##### 5.1 它不是必加
不是所有 CNN 都一定要加 Dropout。

尤其在一些现代 CNN 里，如果：
* 数据量较大
* 使用了 BatchNorm
* 结构设计较合理

有时 Dropout 的使用会减少。

##### 5.2 使用总结
* 经典 CNN：FC 部分常加 Dropout
* 卷积部分：可以加，但更谨慎
* 现代网络中：是否使用要看整体结构

#### 6. CNN 中的 Batch Normalization

##### 6.1 核心作用
BatchNorm 在 CNN 中依然有两个非常重要的作用：
* 让训练更稳定
* 让收敛更快

同时它也常常带来一定的正则化效果。

##### 6.2 在 CNN 中特别常见
相比 Dropout，BatchNorm 在 CNN 里通常更常见、也更核心。

因为卷积网络层数一多，训练稳定性就变得非常重要。

所以在很多 CNN 结构中，

你会经常看到这样的组合：

`Conv → BatchNorm → ReLU`

这几乎是一个非常经典的搭配。

#### 7. CNN 中 BatchNorm 和 MLP 中的区别

##### 7.1 基本思想一样
和 MLP 一样，CNN 中的 BatchNorm 也是对激活值做归一化处理，
让不同 batch 的分布更稳定一些。

##### 7.2 但使用的是 BatchNorm2d
在 MLP 中，我们更常见的是：

`nn.BatchNorm1d`

而在 CNN 中，因为处理的是四维张量：

`[batch_size, channels, height, width]`

所以通常使用的是：

`nn.BatchNorm2d`

#### 8. CNN 中 BatchNorm 一般放在哪里

##### 8.1 最常见的位置
最经典的写法通常是：

`Conv → BatchNorm → ReLU`

也就是说：
* 先卷积
* 再做 BN
* 最后激活

这是最常见、最值得优先记住的顺序。

##### 8.2 为什么这样放
因为卷积层输出后，

先通过 BatchNorm 让分布更稳定，

再进入 ReLU，通常训练效果会更好、更稳定。

所以在 CNN 中，你以后看到这种结构时要非常熟悉：

`卷积 + BN + ReLU`

#### 9. PyTorch 中 CNN 里的 BatchNorm 写法

这里的 16 表示：

这一卷积层输出通道数是 16，所以 BatchNorm2d 也处理 16 个通道。

这个结构就是很典型的：

`Conv → BN → ReLU → Pool`

In [3]:
class CNNWithDropout(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

#### 10. CNN 中如何理解 BatchNorm 的实际价值
**1️⃣ 让每层训练更稳定**

卷积层一层层堆叠后，

中间特征分布可能不断变化。

BN 可以帮助缓解这种不稳定，让训练更顺。

---

**2️⃣ 通常比 Dropout 更“基础”**

在很多 CNN 中，BatchNorm 的使用频率往往比 Dropout 更高。

尤其是现代 CNN，BN 几乎已经是常规组件之一。

在 CNN 里，BN 往往比 Dropout 更常见。

---

**3️⃣ 与现代网络搭配非常常见**

特别是在我们前面学过的现代风格 CNN 中：
* stride > 1 的卷积
* GAP 结构
* 更深的卷积堆叠

这些结构里都很常看到 BatchNorm。

#### 11. Dropout 和 BatchNorm 在 CNN 中怎么搭配理解

##### 11.1 不是二选一
它们不是互相排斥的。

在同一个 CNN 中，完全可以同时出现：
* 前面卷积块中使用 BatchNorm
* 后面 FC 部分使用 Dropout

这是非常常见的搭配方式。

##### 11.2 一个常见组合思路
* 卷积特征提取部分：更常用 BatchNorm
* 分类头 / FC 部分：更常见 Dropout

##### 11.3 完整案例
这个例子里：
* 卷积部分用了 BatchNorm2d
* 全连接部分用了 Dropout

这就是非常典型的 CNN 正则化 / 稳定训练写法。

In [4]:
class CNNWithBNAndDropout(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x